# D4 — QLoRA Fine-tuning on Qwen2.5-1.5B-Instruct

**Run this on Colab Free (T4 GPU).**

## What this notebook does
1. Installs training dependencies.
2. Mounts Google Drive (so the adapter survives runtime restarts).
3. Uploads `data/qa_train.jsonl` and `data/qa_eval.jsonl` from your machine.
4. Loads Qwen2.5-1.5B-Instruct in 4-bit (NF4) quantization.
5. Attaches a LoRA adapter (r=16, alpha=32).
6. Trains for 3 epochs with SFTTrainer.
7. Saves the adapter to Drive at `MyDrive/d4_qlora_adapter/`.

## Before you start
- Runtime → Change runtime type → **T4 GPU**.
- Have `data/qa_train.jsonl` and `data/qa_eval.jsonl` ready to upload.

## 1. Install dependencies

In [ ]:
!pip uninstall -y bitsandbytes triton
!pip install -q -U \
    "bitsandbytes>=0.45.0" \
    "transformers>=4.46.0" \
    "peft>=0.13.0" \
    "trl>=0.12.0" \
    "accelerate>=1.0.0" \
    "datasets>=3.0.0"

Found existing installation: bitsandbytes 0.43.3
Uninstalling bitsandbytes-0.43.3:
  Successfully uninstalled bitsandbytes-0.43.3
Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q -U \
    "transformers==4.44.2" \
    "peft==0.12.0" \
    "trl==0.10.1" \
    "accelerate==0.33.0" \
    "bitsandbytes==0.43.3" \
    "datasets==2.21.0" \
    "sentencepiece" \
    "protobuf"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.64 GB


## 2. Mount Google Drive

Colab kills your runtime after a few hours of inactivity, but Drive persists. We'll save the adapter there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = "/content/drive/MyDrive/d4_qlora_adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Adapter will be saved to:", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adapter will be saved to: /content/drive/MyDrive/d4_qlora_adapter


## 3. Upload your data files

Click the folder icon on the left → upload `qa_train.jsonl` and `qa_eval.jsonl`.
Or run the cell below to use the upload widget.

In [ ]:
from google.colab import files
print("Upload qa_train.jsonl and qa_eval.jsonl:")
uploaded = files.upload()

TRAIN_PATH = "qa_train.jsonl"
EVAL_PATH  = "qa_eval.jsonl"

assert os.path.exists(TRAIN_PATH), "qa_train.jsonl missing"
assert os.path.exists(EVAL_PATH),  "qa_eval.jsonl missing"

import json
with open(TRAIN_PATH) as f:
    n_train = sum(1 for _ in f)
with open(EVAL_PATH) as f:
    n_eval = sum(1 for _ in f)
print(f"train: {n_train} examples")
print(f"eval:  {n_eval} examples")

Upload qa_train.jsonl and qa_eval.jsonl:


Saving qa_eval.jsonl to qa_eval (1).jsonl
Saving qa_train.jsonl to qa_train (1).jsonl
train: 174 examples
eval:  44 examples


## 4. Load base model in 4-bit

Qwen2.5-1.5B-Instruct is open — no HF login needed.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False  # required for training

print("Base model loaded.")
print("Memory footprint:", round(model.get_memory_footprint() / 1e9, 2), "GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded.
Memory footprint: 1.12 GB


## 5. Attach LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


## 6. Format dataset with Qwen chat template

Each example becomes a conversation: system prompt → user question + context → assistant answer.
This matches how the model will be used in `/ask`.

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You answer questions from scientific papers. "
    "Use ONLY the provided context. Cite sources inline as [1], [2], etc. "
    "Be concise (1-3 sentences). If the answer is not in the context, say so."
)

def to_chat(example):
    user_msg = f"Question: {example['question']}\n\nContext:\n[1] {example['context']}"
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": example["answer"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

ds = load_dataset("json", data_files={"train": TRAIN_PATH, "eval": EVAL_PATH})
ds = ds.map(to_chat, remove_columns=ds["train"].column_names)

print("Sample formatted example:\n")
print(ds["train"][0]["text"][:600])
print("...")

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/174 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Sample formatted example:

<|im_start|>system
You answer questions from scientific papers. Use ONLY the provided context. Cite sources inline as [1], [2], etc. Be concise (1-3 sentences). If the answer is not in the context, say so.<|im_end|>
<|im_start|>user
Question: What does the Neural Feature Ansatz (NFA) claim about Gram matrices of weights in a neural network layer?

Context:
[1] A Related Works & Limitations A.1 Comprehensive Related Works As feature learning is one of the distinguishing advantages of neural networks over classical machine learning models (such as kernel machines), an emerging body of work has f
...


## 7. Train with SFTTrainer

In [ ]:
import os, torch, gc

# Free whatever's left from the failed run
gc.collect()
torch.cuda.empty_cache()

# Reduce memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,           # ↓ was 2
    per_device_eval_batch_size=1,            # NEW — eval was using a bigger default
    gradient_accumulation_steps=16,          # ↑ keep effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_length=768,                          # ↓ was 1024 — your contexts are short anyway
    gradient_checkpointing=True,             # NEW — trades ~20% speed for big memory savings
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds["train"],
    eval_dataset=ds["eval"],
    args=training_args,
)

print("Starting training...")
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training...


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,2.422447,2.271672,2.266409,99719.000000,0.555352
2,2.274494,2.245523,2.232900,199438.000000,0.558752
3,2.295206,2.241476,2.227659,299157.000000,0.560448


TrainOutput(global_step=33, training_loss=2.3453001976013184, metrics={'train_runtime': 1606.8781, 'train_samples_per_second': 0.325, 'train_steps_per_second': 0.021, 'total_flos': 2359808007312384.0, 'train_loss': 2.3453001976013184, 'epoch': 3.0})

In [ ]:
# ============================================================
# D4 — AutoML hyperparameter search with Optuna (5 trials)
# Connects back to D1's Optuna methodology
# ============================================================

!pip install -q optuna

import os, gc, json, torch, optuna
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- Free the previous model from memory --------------------
try:
    del trainer, model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

# ---- Search space -------------------------------------------
# We tune the 2 highest-leverage knobs from D4 training experience:
#   - learning_rate (log uniform)
#   - lora_r        (categorical)
# Each trial trains 1 epoch only as a proxy for full convergence.

TRIAL_DIR = "/content/optuna_trials"
os.makedirs(TRIAL_DIR, exist_ok=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

def build_fresh_model(lora_r, lora_alpha, lora_dropout):
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base)
    cfg = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    return get_peft_model(base, cfg)


def objective(trial):
    # ---- suggest hyperparams ----
    lr     = trial.suggest_float("lr", 1e-4, 5e-4, log=True)
    lora_r = trial.suggest_categorical("lora_r", [16, 32, 64])

    # alpha tied to 2×r (community heuristic that works well)
    lora_alpha = 2 * lora_r

    print(f"\n{'='*60}")
    print(f"Trial {trial.number}: lr={lr:.2e}, lora_r={lora_r}, alpha={lora_alpha}")
    print(f"{'='*60}")

    # ---- build fresh model for this trial ----
    m = build_fresh_model(lora_r, lora_alpha, 0.05)

    trial_out = f"{TRIAL_DIR}/trial_{trial.number}"
    args = SFTConfig(
        output_dir=trial_out,
        num_train_epochs=1,                       # 1 epoch as proxy
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=10,
        save_strategy="no",                       # don't save trial checkpoints
        eval_strategy="epoch",
        bf16=True,
        optim="paged_adamw_8bit",
        report_to="none",
        max_length=768,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

    tr = SFTTrainer(
        model=m,
        train_dataset=ds["train"],
        eval_dataset=ds["eval"],
        args=args,
    )
    tr.train()
    eval_metrics = tr.evaluate()
    eval_loss = eval_metrics["eval_loss"]
    print(f"  → eval_loss = {eval_loss:.4f}")

    # ---- cleanup before next trial ----
    del tr, m
    gc.collect()
    torch.cuda.empty_cache()

    return eval_loss

# ---- Run the study ------------------------------------------
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="d4_qlora_tuning",
)
study.optimize(objective, n_trials=5)

# ---- Report -------------------------------------------------
print("\n" + "="*60)
print("OPTUNA STUDY COMPLETE")
print("="*60)
print(f"Best eval_loss: {study.best_value:.4f}")
print(f"Best params:    {study.best_params}")
print("\nAll trials:")
for t in study.trials:
    print(f"  Trial {t.number}: lr={t.params['lr']:.2e}, "
          f"r={t.params['lora_r']}, eval_loss={t.value:.4f}")

# Save study for the tuning card
with open("/content/drive/MyDrive/d4_qlora_adapter/optuna_study.json", "w") as f:
    json.dump({
        "best_value":  study.best_value,
        "best_params": study.best_params,
        "trials": [
            {"number": t.number, "params": t.params, "value": t.value}
            for t in study.trials
        ],
    }, f, indent=2)
print("\nStudy saved to MyDrive/d4_qlora_adapter/optuna_study.json")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 32.0 MB/s eta 0:00:00


[I 2026-06-07 21:46:23,428] A new study created in memory with name: d4_qlora_tuning


GPU memory free: 4.45 GB

Trial 0: lr=1.83e-04, lora_r=16, alpha=32


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


...

[I 2026-06-07 22:16:24,917] Trial 2 finished with value: 2.331230878829956 and parameters: {'lr': 0.000263124545105745, 'lora_r': 64}. Best is trial 2 with value: 2.331230878829956.



Trial 3: lr=3.82e-04, lora_r=16, alpha=32


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
[W 2026-06-07 22:16:46,340] Trial 3 failed with parameters: {'lr': 0.0003818145165896873, 'lora_r': 16} because of the following error: OutOfMemoryError('CUDA out of memory. Tried to allocate 446.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 433.81 MiB is free. Including non-PyTorch memory, this process has 14.14 GiB memory in use. Of the allocated me

OutOfMemoryError: CUDA out of memory. Tried to allocate 446.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 433.81 MiB is free. Including non-PyTorch memory, this process has 14.14 GiB memory in use. Of the allocated memory 13.69 GiB is allocated by PyTorch, and 321.44 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# Check what we have from the partial Optuna run
print("Completed trials:")
print(f"  Total: {len(study.trials)}")
print(f"  Succeeded: {sum(1 for t in study.trials if t.value is not None)}")
print(f"  Failed: {sum(1 for t in study.trials if t.value is None)}\n")

# Show all completed trials sorted by eval_loss
done = [t for t in study.trials if t.value is not None]
done.sort(key=lambda t: t.value)

print("Trials that succeeded (sorted by eval_loss):")
for t in done:
    print(f"  Trial {t.number}: lr={t.params['lr']:.2e}, "
          f"r={t.params['lora_r']}, eval_loss={t.value:.4f}")

if done:
    best = done[0].params
    print(f"\n✅ Best so far: lr={best['lr']:.2e}, lora_r={best['lora_r']}")

    # Save what we have
    import json
    with open("/content/drive/MyDrive/d4_qlora_adapter/optuna_study.json", "w") as f:
        json.dump({
            "note": "Partial study — 3/5 trials completed before T4 OOM",
            "best_value": done[0].value,
            "best_params": best,
            "trials": [{"number": t.number, "params": t.params,
                        "value": t.value} for t in done],
        }, f, indent=2)
    print("Saved partial study to Drive.")
else:
    print("No trials succeeded. Falling back to manual hyperparams.")

Completed trials:
  Total: 4
  Succeeded: 3
  Failed: 1

Trials that succeeded (sorted by eval_loss):
  Trial 2: lr=2.63e-04, r=64, eval_loss=2.3312
  Trial 1: lr=1.29e-04, r=64, eval_loss=2.4588
  Trial 0: lr=1.83e-04, r=16, eval_loss=2.6325

✅ Best so far: lr=2.63e-04, lora_r=64
Saved partial study to Drive.


In [ ]:
import json, os, gc, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Pull winning hyperparams
with open("/content/drive/MyDrive/d4_qlora_adapter/optuna_study.json") as f:
    best = json.load(f)["best_params"]
print(f"Using AutoML-selected: lr={best['lr']:.2e}, lora_r={best['lora_r']}\n")

# Fresh model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base.config.use_cache = False
base = prepare_model_for_kbit_training(base)

model = get_peft_model(base, LoraConfig(
    r=best["lora_r"],
    lora_alpha=2 * best["lora_r"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
))
model.print_trainable_parameters()

trainer = SFTTrainer(
    model=model,
    train_dataset=ds["train"],
    eval_dataset=ds["eval"],
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=best["lr"],
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=5,
        save_strategy="epoch",
        eval_strategy="epoch",
        bf16=True,
        optim="paged_adamw_8bit",
        report_to="none",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_length=768,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    ),
)
print("\nStarting final training with AutoML-selected config...")
trainer.train()

ADAPTER_DIR = f"{OUTPUT_DIR}/final"
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\n✅ Final adapter saved to {ADAPTER_DIR}")

Using AutoML-selected: lr=2.63e-04, lora_r=64



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 17,432,576 || all params: 1,561,146,880 || trainable%: 1.1167


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting final training with AutoML-selected config...


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,2.472985,2.260624,2.226079,99719.000000,0.556447
2,2.231502,2.217425,2.216111,199438.000000,0.560069
3,2.221725,2.210831,2.200812,299157.000000,0.562111



✅ Final adapter saved to /content/drive/MyDrive/d4_qlora_adapter/final


## 8. Save the adapter

This saves just the LoRA adapter weights (small — ~10–30MB) to Drive.

In [ ]:
ADAPTER_DIR = f"{OUTPUT_DIR}/final"
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
print("Saved files:")
for f in os.listdir(ADAPTER_DIR):
    size_mb = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.2f} MB)")

Saved files:
  README.md  (0.01 MB)
  adapter_model.safetensors  (69.76 MB)
  adapter_config.json  (0.00 MB)
  chat_template.jinja  (0.00 MB)
  tokenizer_config.json  (0.00 MB)
  tokenizer.json  (11.42 MB)
  training_args.bin  (0.01 MB)


## 9. Quick smoke test — does the tuned model produce reasonable output?

In [ ]:
model.eval()

test_q = "What is the main computational drawback of the RLPD algorithm?"
test_ctx = ("RLPD requires storing the full replay buffer in memory, which scales "
            "linearly with the number of environment steps. For long-horizon tasks "
            "this becomes prohibitive.")

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": f"Question: {test_q}\n\nContext:\n[1] {test_ctx}"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("GENERATED:")
print(generated)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


GENERATED:
The main computational drawback of the RLPD algorithm is that it requires storing the full replay buffer in memory, which scales linearly with the number of environment steps and can become prohibitively large for long-horizon tasks. [1]


## Done!

**Next steps** — outside this notebook:

1. **Run the merge notebook** (`merge_and_export.ipynb`) to merge the adapter into the base model and prepare for GGUF conversion.
2. **Download the merged model** from Drive.
3. **Convert to GGUF** and load into Ollama using the provided `Modelfile`.

Adapter is now at: `MyDrive/d4_qlora_adapter/final/`

Also: take a screenshot of the training loss curve from the cell above — you'll need it for the tuning card.